# 04 — Retrieval Quality

Evaluate whether query_items-style queries return the right videos, independent of the agent.

Tests both structured filtering (topic, affect, etc.) and embedding-based semantic search.

In [ ]:
import sys, os, json
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if "workbench" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / "workbench" / ".env")

import numpy as np
import pandas as pd

# Load sample data
samples_path = REPO_ROOT / "workbench" / "data" / "sample-videos.json"
samples = json.loads(samples_path.read_text())

if not samples:
    print("sample-videos.json is empty. Generate test data first:")
    print("  python workbench/scripts/generate_test_data.py 'diverse tiktok videos' --count 20")
else:
    df = pd.DataFrame(samples)
    # Flatten expected labels into columns
    if "expected" in df.columns:
        expected_df = pd.json_normalize(df["expected"])
        df = pd.concat([df.drop(columns=["expected"]), expected_df], axis=1)
    print(f"Loaded {len(df)} sample videos")
    df.head()

In [ ]:
def query_items_inmemory(
    df: pd.DataFrame,
    search_text: str | None = None,
    hashtag: str | None = None,
    creator: str | None = None,
    topic: str | None = None,
    affect: str | None = None,
    genre: str | None = None,
    media_type: str | None = None,
    limit: int = 20,
) -> pd.DataFrame:
    """In-memory replica of query_items filtering logic."""
    result = df.copy()
    
    if search_text:
        q = search_text.lower()
        mask = result["caption"].fillna("").str.lower().str.contains(q, na=False)
        if "subtitle" in result.columns:
            mask |= result["subtitle"].fillna("").str.lower().str.contains(q, na=False)
        result = result[mask]
    
    if hashtag:
        result = result[result["hashtags"].apply(
            lambda tags: hashtag.lower() in [t.lower().lstrip("#") for t in (tags or [])]
        )]
    
    if creator:
        result = result[result["creator"].fillna("").str.lower().str.contains(creator.lower(), na=False)]
    
    if topic and "topic" in result.columns:
        result = result[result["topic"] == topic]
    if affect and "affect" in result.columns:
        result = result[result["affect"] == affect]
    if genre and "genre" in result.columns:
        result = result[result["genre"] == genre]
    if media_type and "media_type" in result.columns:
        result = result[result["media_type"] == media_type]
    
    return result.head(limit)

In [ ]:
# Define test queries with expected results
# Fill in expected_ids after populating sample-videos.json
test_queries = [
    {
        "name": "funny cooking videos",
        "filters": {"topic": "food", "affect": "funny"},
        "expected_ids": [],  # Fill with IDs from sample-videos.json
    },
    {
        "name": "tutorial videos",
        "filters": {"genre": "tutorial"},
        "expected_ids": [],
    },
    {
        "name": "text search: pasta",
        "filters": {"search_text": "pasta"},
        "expected_ids": [],
    },
]

if samples:
    for tq in test_queries:
        results = query_items_inmemory(df, **tq["filters"])
        returned_ids = set(results["id"].tolist()) if "id" in results.columns else set()
        expected_ids = set(tq["expected_ids"])
        
        if expected_ids:
            precision = len(returned_ids & expected_ids) / len(returned_ids) if returned_ids else 0
            recall = len(returned_ids & expected_ids) / len(expected_ids) if expected_ids else 0
            print(f"{tq['name']:30s} returned={len(returned_ids):3d}  precision={precision:.0%}  recall={recall:.0%}")
        else:
            print(f"{tq['name']:30s} returned={len(results):3d}  (no expected_ids set — fill in to measure precision/recall)")
        
        if not results.empty:
            display(results[[c for c in ["id", "caption", "topic", "affect", "genre"] if c in results.columns]].head(5))

In [ ]:
# Embedding-based semantic search (requires embeddings)
# To compute embeddings for the sample set, use OpenAI:

from openai import OpenAI

def compute_embedding(text: str) -> list[float]:
    """Compute embedding using the same model as the pipeline."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Example: compute similarity between two queries
if os.environ.get("OPENAI_API_KEY"):
    e1 = compute_embedding("funny cat videos")
    e2 = compute_embedding("hilarious pet content")
    e3 = compute_embedding("mortgage interest rates")
    print(f"'funny cat videos' vs 'hilarious pet content': {cosine_similarity(e1, e2):.3f}")
    print(f"'funny cat videos' vs 'mortgage interest rates': {cosine_similarity(e1, e3):.3f}")
else:
    print("OPENAI_API_KEY not set — skipping embedding demo")